# 02c - 从TRIZ-raw构建预训练语料库

将 `TRIZ-raw/` 中的 PDF/DOCX/PPTX/DOC 材料提取为结构化 JSONL 语料。

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

from pathlib import Path
from config import CORPUS_CONFIG
from utils.corpus_builder import build_corpus

print('CORPUS_CONFIG:', CORPUS_CONFIG['raw_dir'])

In [ ]:
# 检查 TRIZ-raw 目录
raw_dir = Path(CORPUS_CONFIG['raw_dir'])
print(f'Raw dir exists: {raw_dir.exists()}')
print(f'Files found: {len(list(raw_dir.rglob("*")))}')

from collections import Counter
exts = Counter([p.suffix.lower() for p in raw_dir.rglob("*") if p.is_file()])
print('Extension counts:')
for ext, count in exts.most_common():
    print(f'  {ext}: {count}')

In [ ]:
# 构建语料库
stats = build_corpus(
    raw_dir=CORPUS_CONFIG['raw_dir'],
    output_dir=CORPUS_CONFIG['output_dir'],
    output_filename=CORPUS_CONFIG['output_filename'],
    stats_filename=CORPUS_CONFIG['stats_filename'],
    failed_files_filename=CORPUS_CONFIG['failed_files_filename'],
    chunk_target_tokens=CORPUS_CONFIG['chunk']['target_tokens'],
    chunk_max_tokens=CORPUS_CONFIG['chunk']['max_tokens'],
    chars_per_token=CORPUS_CONFIG['chunk']['chars_per_token'],
    min_chars=CORPUS_CONFIG['quality_gates']['min_chars'],
    deduplicate=CORPUS_CONFIG['quality_gates']['deduplicate'],
    ocr_enabled=CORPUS_CONFIG['ocr']['enabled'],
    ocr_min_text_chars=CORPUS_CONFIG['ocr']['min_text_chars'],
    resume=True,
)

print('Build complete.')
print(stats)

In [ ]:
# 查看统计和样本
import json
output_path = Path(CORPUS_CONFIG['output_dir']) / CORPUS_CONFIG['output_filename']
with open(output_path, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f][:5]

for r in records:
    print(f"ID: {r['id']}")
    print(f"Category: {r['metadata']['category']}")
    print(f"Tokens: {r['metadata']['token_count']}")
    print(f"Text preview: {r['text'][:200]}...")
    print('-' * 40)

In [ ]:
# 检查失败文件
failed_path = Path(CORPUS_CONFIG['output_dir']) / CORPUS_CONFIG['failed_files_filename']
if failed_path.exists():
    with open(failed_path, 'r', encoding='utf-8') as f:
        failed = json.load(f)
    print(f'Failed files: {len(failed)}')
    for item in failed[:10]:
        print(item)
else:
    print('No failed files.')